# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Machine Learning: Alternating Least Squares (ALS)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [16]:
from SparkUtils import SparkUtils

from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import explode

In [3]:

MASTER_URL = "spark://spark-master:7077"
APP_NAME = "Lab 12: Alternating Least Squares (ALS) for Collaborative Filtering"

spark = SparkUtils(MASTER_URL, APP_NAME)._spark

spark

# Example 1: Songs recommednation

In [ ]:
# Sample user-song interaction data
data = [(1, 1, 4),
        (1, 2, 5),
        (1, 5, 5),
        (2, 2, 3),
        (2, 3, 4),
        (2, 4, 3),
        (3, 1, 2),
        (3, 3, 5),
        (3, 5, 1)]
  
# Define schema for the DataFrame
schema = SparkUtils.generate_schema([("user_id", "int"), ("song_id", "int"), ("rating", "int")])

# Create DataFrame for interactions
interactions_df = spark.createDataFrame(data, schema)

interactions_df.show()

+-------+-------+------+
|user_id|song_id|rating|
+-------+-------+------+
|      1|      1|     4|
|      1|      2|     5|
|      1|      5|     5|
|      2|      2|     3|
|      2|      3|     4|
|      2|      4|     3|
|      3|      1|     2|
|      3|      3|     5|
|      3|      5|     1|
+-------+-------+------+



In [5]:
print(f"Number of items o canciones (n):{interactions_df.groupBy('song_id').count().count()}")
print(f"Number of users (m):{interactions_df.groupBy('user_id').count().count()}")

Number of items o canciones (n):5
Number of users (m):3


In [8]:
als = ALS(
    userCol="user_id", 
    itemCol="song_id", 
    ratingCol="rating", 
    maxIter=10, 
    regParam=0.1, 
    rank=5, # Controls the dimensionality of the latent vector space for 
            # users and items.
    coldStartStrategy="drop"  # Avoids NaN predictions
)

In [9]:
model = als.fit(interactions_df)

print("Recommendation system generated successfully")

Recommendation system generated successfully


In [10]:
# Generate recommendations for each user
user_recommendations = model.recommendForAllUsers(numItems=3)

# Show recommendations
user_recommendations.show(truncate=False)

+-------+-----------------------------------------------+
|user_id|recommendations                                |
+-------+-----------------------------------------------+
|1      |[{2, 4.952161}, {5, 4.85698}, {1, 3.9416614}]  |
|2      |[{3, 3.947534}, {2, 2.9674487}, {4, 2.9087834}]|
|3      |[{3, 4.8383236}, {4, 3.171386}, {2, 2.3287745}]|
+-------+-----------------------------------------------+



In [17]:
songs = [
    (1, "song a"),
    (2, "song b"),
    (3, "song c"),
    (4, "song d"),
    (5, "song e")
]

songs_schema = SparkUtils.generate_schema([("song_id", "int"), ("title", "string")])

songs_df = spark.createDataFrame(songs, songs_schema)

In [18]:
# Explode recommendations for easier reading
recommendations = user_recommendations.select("user_id", explode("recommendations").alias("rec"))
recommendations = recommendations.join(songs_df, recommendations.rec.song_id == songs_df.song_id).select("user_id", "title", "rec.rating")

# Show user-song recommendations with titles
recommendations.show(truncate=False)

+-------+------+---------+
|user_id|title |rating   |
+-------+------+---------+
|1      |song a|3.9416614|
|3      |song b|2.3287745|
|2      |song b|2.9674487|
|1      |song b|4.952161 |
|3      |song c|4.8383236|
|2      |song c|3.947534 |
|1      |song e|4.85698  |
|3      |song d|3.171386 |
|2      |song d|2.9087834|
+-------+------+---------+



In [19]:
predictions = model.transform(interactions_df)

predictions.show(truncate=False)

+-------+-------+------+----------+
|user_id|song_id|rating|prediction|
+-------+-------+------+----------+
|1      |1      |4     |3.9416614 |
|1      |2      |5     |4.952161  |
|1      |5      |5     |4.85698   |
|2      |2      |3     |2.9674487 |
|3      |1      |2     |1.9654449 |
|3      |3      |5     |4.8383236 |
|3      |5      |1     |1.0485418 |
|2      |3      |4     |3.947534  |
|2      |4      |3     |2.9087834 |
+-------+-------+------+----------+



In [20]:
# Evaluate the Recommendation System
# Set up evaluator to compute RMSE
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="rating", 
    predictionCol="prediction"
)

# Calculate RMSE
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error (RMSE) = {rmse}")

Root-mean-square error (RMSE) = 0.08690293822180356


# Lab 12: Building a Recommendation System with ALS 

In [31]:
movies_ratings_path = "/opt/spark/work-dir/data/ml/als"

movies_ratings_schema = SparkUtils.generate_schema([("user_id", "int"), ("movie_id", "int"), ("rating", "int"),("timestamp", "int")])

# Source https://github.com/databricks/Spark-The-Definitive-Guide/blob/master/data/sample_movielens_ratings.txt
movies_ratings_df = spark.read \
    .option("header", "false") \
    .option("delimiter", "::") \
    .schema(movies_ratings_schema) \
    .csv(movies_ratings_path)

movies_ratings_df.printSchema()

movies_ratings_df.show(3)

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)

+-------+--------+------+----------+
|user_id|movie_id|rating| timestamp|
+-------+--------+------+----------+
|      0|       2|     3|1424380312|
|      0|       3|     1|1424380312|
|      0|       5|     2|1424380312|
+-------+--------+------+----------+
only showing top 3 rows


## Create & Train the ML Model

In [32]:
print(f"Number of items or movies (n):{movies_ratings_df.groupBy('movie_id').count().count()}")

print(f"Number of users (m):{movies_ratings_df.groupBy('user_id').count().count()}")

Number of items or movies (n):100
Number of users (m):30


In [33]:
als = ALS(
    userCol="user_id", 
    itemCol="movie_id", 
    ratingCol="rating", 
    maxIter=10, 
    regParam=0.1, 
    rank=5, # Controls the dimensionality of the latent vector space for 
            # users and items.
    coldStartStrategy="drop"  # Avoids NaN predictions
)

In [34]:
model = als.fit(movies_ratings_df)

print("Recommendation system generated successfully")

Recommendation system generated successfully


## Persist the model

In [35]:
model_path = "/opt/spark/work-dir/data/mlmodels/als/output"

model.save(model_path)

print(f"Model saved to: {model_path}")

Model saved to: /opt/spark/work-dir/data/mlmodels/als/output


## Predictions

In [39]:
# Generate top 3 movie recommendations for each user
user_recommendations = model.recommendForAllUsers(numItems=4)

user_recommendations.show(truncate=False)

+-------+--------------------------------------------------------------------+
|user_id|recommendations                                                     |
+-------+--------------------------------------------------------------------+
|0      |[{92, 2.6200228}, {2, 2.355422}, {62, 2.242904}, {93, 2.163558}]    |
|10     |[{92, 2.795528}, {2, 2.6930857}, {49, 2.6012158}, {93, 2.5987248}]  |
|20     |[{22, 3.5411158}, {68, 3.1099432}, {94, 3.08887}, {77, 3.055102}]   |
|1      |[{22, 2.881381}, {68, 2.6039166}, {77, 2.5396843}, {62, 2.518314}]  |
|11     |[{32, 5.034642}, {30, 4.7379417}, {18, 4.646621}, {27, 4.511575}]   |
|21     |[{29, 4.3013725}, {52, 4.2179327}, {76, 3.6884913}, {63, 3.460023}] |
|22     |[{51, 4.4359684}, {75, 4.4221845}, {74, 4.10294}, {22, 4.093457}]   |
|2      |[{93, 4.2507396}, {83, 4.171638}, {8, 4.0521445}, {39, 3.7443476}]  |
|12     |[{46, 5.7872624}, {55, 4.7876105}, {49, 4.5210724}, {90, 4.24609}]  |
|23     |[{46, 5.553124}, {55, 4.700608}, {90, 4.670

In [40]:
predictions = model.transform(movies_ratings_df)

predictions.show(truncate=False)

+-------+--------+------+----------+----------+
|user_id|movie_id|rating|timestamp |prediction|
+-------+--------+------+----------+----------+
|22     |0       |1     |1424380312|0.9739337 |
|22     |3       |2     |1424380312|1.6246576 |
|22     |5       |2     |1424380312|2.0648289 |
|22     |6       |2     |1424380312|2.2927299 |
|22     |9       |1     |1424380312|1.5843192 |
|22     |10      |1     |1424380312|1.4450505 |
|22     |11      |1     |1424380312|1.2751293 |
|22     |13      |1     |1424380312|1.6177039 |
|22     |14      |1     |1424380312|1.3729261 |
|22     |16      |1     |1424380312|0.6965815 |
|22     |18      |3     |1424380312|3.0357275 |
|22     |19      |1     |1424380312|1.4458134 |
|22     |22      |5     |1424380312|4.093457  |
|22     |25      |1     |1424380312|0.9862013 |
|22     |26      |1     |1424380312|1.1323587 |
|22     |29      |3     |1424380312|3.2179248 |
|22     |30      |5     |1424380312|3.9856434 |
|22     |32      |4     |1424380312|3.16

## Test ML Model

In [41]:
# Evaluate the Recommendation System
# Set up evaluator to compute RMSE
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="rating", 
    predictionCol="prediction"
)

# Calculate RMSE
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error (RMSE) = {rmse}")

Root-mean-square error (RMSE) = 0.5698671295341873


In [42]:
spark.stop()